In [1]:
import pandas as pd

/Users/maryamzakiyya/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [6]:
df = pd.read_csv('all_files.csv')

In [7]:
df

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
2,2,11-13-20 Kaur and Singh.pdf,"Comments Received After September 30, 2020",1,0.000008,"November 13th, 2020\n\nDear Superintendent Thu..."
3,3,11-19-20 Lamont.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sami Lamont Sent: Wednesday, November 18..."
4,4,11-18-20 Jensen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sine Hwang Jensen Sent: Tuesday, Novembe..."
...,...,...,...,...,...,...
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [5]:
df_duplicate = df[df['submissions'] > 1]

In [6]:
df_duplicate['submissions'].sum()

66840

In [7]:
df['submissions'].sum()

74466

In [12]:
df_duplicate.sort_values('submissions', ascending=False)

,Unnamed: 0,file_name,folder_name,submissions,weight,text
3897,3897,8-15-19 Israeli American Council.pdf,First Field Review (June - August 2019),14275,0.256152,The California Department of Education Receive...
7056,7057,7-8-20 Group Letter Arab American Studies.pdf,Comments Received After Field Review,10000,0.173633,The California Department of Education receive...
1710,1710,8-26-20 Group Letter Armenian American.pdf,Comments Received After Field Review,6000,0.104180,The California Department of Education receive...
1088,1088,12-17-20 Group Letter Arab Americans 3.pdf,Third Field Review (Dec 2020 - Jan 2021),5000,0.045296,The California Department of Education receive...
160,160,11-11-20 Group Letter Arab Americans 2.pdf,"Comments Received After September 30, 2020",4600,0.036716,The California Department of Education receive...
...,...,...,...,...,...,...
6259,6260,9-30-20 Group Letter Jewish Americans 5.pdf,Second Field Review (Sept 2020),7,0.000078,The California Department of Education receive...
1041,1041,12-8-20 Group Letter Arab American.pdf,Third Field Review (Dec 2020 - Jan 2021),7,0.000063,The California Department of Education receive...
6246,6247,9-21-20 Group Letter Arab American Studies 3.pdf,Second Field Review (Sept 2020),5,0.000055,The California Department of Education receive...
30,30,11-12-20 Group Letter Critical Race Theory.pdf,"Comments Received After September 30, 2020",5,0.000040,The California Department of Education receive...


In [8]:
import re

def is_legible(text):
    """Return True if text seems legible and not random gibberish."""
    if not isinstance(text, str) or len(text.strip()) < 10:
        return False

    # Must contain at least some real words (a-z)
    if not re.search(r"[a-zA-Z]", text):
        return False

    # Remove if text has too many symbols or uppercase nonsense
    letters = re.findall(r"[A-Za-z\s]", text)
    ratio = len(letters) / len(text)
    if ratio < 0.5:  # more than half non-alphabetic = likely gibberish
        return False

    return True

# Apply filter
df_illegible = df[df["text"].apply(is_legible)]


In [10]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993225 sha256=5813f5da3ebd51cca55f209c3ba962b108bca0dfbfa759b3404fb6905ca1c850
  Stored in directory: /Users/maryamzakiyya/Library/Caches/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
DEPRECATION: textract 1.6.5 has a non-standard dependency specifier extract-msg<=0.29.*. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of textract or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [11]:
import re
from langdetect import detect, LangDetectException

def is_illegible(text):
    """Return True if text is likely gibberish or illegible."""
    if not isinstance(text, str) or len(text.strip()) < 10:
        return True

    # If there's too much noise (symbols, numbers, etc.)
    ratio_letters = len(re.findall(r"[A-Za-z\s]", text)) / len(text)
    if ratio_letters < 0.5:
        return True

    # Try language detection
    try:
        lang = detect(text)
        # If language can't be detected or isn't English, flag it
        if lang != "en":
            return True
    except LangDetectException:
        return True

    return False

# Apply the check
df["illegible"] = df["text"].apply(is_illegible)

# View only illegible rows
illegible_df = df[df["illegible"] == True]

print(f"Found {len(illegible_df)} illegible rows.")
illegible_df.head()


Found 3 illegible rows.


,Unnamed: 0,file_name,folder_name,submissions,weight,text,illegible
2324,2324,8-8-19 kikiassor.pdf,First Field Review (June - August 2019),1,0.000018,"From: kikiassor Sent: Thursday, August 8, 2019...",True
7486,7487,7-27-20 Kantor.pdf,Comments Received After Field Review,1,0.000017,"‫‪From: Bernard Kantor‬‬\n‫‪Sent: Monday, July...",True
7507,7508,ADL.pdf,Comments Received After Field Review,1,0.000017,Th\n\n,True
